# 客服智慧體

## 1. 引言

*結合程式碼執行的規劃*意味著讓 LLM **編寫程式碼並將其作為計劃本身**。  
與純文字或基於 JSON 的計劃相比，這種方法更具表達力和靈活性：程式碼不僅記錄步驟，還能直接執行它們。

在本範例中，你將實踐這一設計模式。  
我們不再讓 LLM 輸出 JSON 計劃並手動執行每一步，而是允許它**編寫 Python 程式碼**來直接表達一個計劃的多步邏輯。透過執行這些程式碼，我們可以自動完成複雜查詢。  

為更具體地展示這一點，我們模擬一個**太陽鏡商店**，包含產品**庫存**與一組**交易**（銷售、退貨、餘額更新）。該示例展示了 LLM 如何生成用於查詢或更新記錄的程式碼，體現這一模式的靈活性。

### 1.1 範例概覽
我們將：
1. 建立簡單的**庫存**與**交易**資料集。  
2. 構建描述資料的**模式塊（schema block）**。  
3. 透過提示讓 LLM **用 Python 程式碼編寫計劃**（並在程式碼註釋中解釋每一步）。  
4. 在沙箱中執行程式碼以獲得答案。  

### 1.2 學習目標

完成本範例後，你將能夠：

- **說明**為何讓模型編寫程式碼（而非 JSON 或純文字計劃）能實現更豐富、更靈活的規劃。  
- **提示** LLM 生成帶有逐步註釋的 Python 程式碼，使其既能文件化也能執行計劃。  
- **安全執行**生成的程式碼於沙箱中，並解釋結果。  

這說明了為何*以程式碼為行動（Code as Action）*能夠優於脆弱的工具鏈與基於 JSON 的規劃方式。

## 2. 環境設定

In [ ]:
# ==== 匯入 ====
from __future__ import annotations
import json
from dotenv import load_dotenv
from openai import OpenAI
import re, io, sys, traceback, json
from typing import Any, Dict, Optional
from tinydb import Query, where

# 工具模組
import utils      # 用於提示/列印的助手函式
import inv_utils  # 用於庫存、交易、模式構建和 TinyDB 資料填充的函式

load_dotenv()
client = OpenAI()

在 `inv_utils` 模組中，包含如下函式：

- `create_inventory()` – 構建太陽鏡庫存。  
- `create_transactions()` – 構建初始交易日誌。  
- `seed_db()` – 將庫存與交易載入到基於 JSON 的儲存中。  
- `build_schema_block()` – 生成在提示中使用的模式描述。  
- 還有諸如 `get_current_balance()` 與 `next_transaction_id()` 的輔助函式——讓 LLM 在庫存與交易之間執行一致的更新。  

### 2.1 建立示例表

我們將使用 **[TinyDB](https://tinydb.readthedocs.io/)** 為太陽鏡商店模擬建立兩個小型表。TinyDB 是一個用純 Python 編寫的輕量級文件型資料庫。  
TinyDB 以 JSON 文件的形式儲存資料，適合小型應用或原型，因為它無需伺服器配置，並且可輕鬆查詢與更新資料。

兩個表為：

- **`inventory_tbl`**：包含產品詳情，如名稱、商品 ID、描述、庫存數量、價格等。  
- **`transactions_tbl`**：以期初餘額開頭，隨後記錄購買、退貨和調整。  

你將使用 `inv_utils` 中的輔助函式生成這些表，並在下面預覽前幾行。

In [ ]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()

現在，你可以透過列印格式化的 JSON 來檢查各表中的記錄：

In [ ]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="庫存表")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="交易表")

如上所示，各表的模式如下：

<div style="border:1px solid #BFDBFE; border-left:6px solid #3B82F6; background:#EFF6FF; border-radius:6px; padding:16px; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif; line-height:1.6; color:#1E3A8A;">

  <h4 style="margin-top:0; color:#1E40AF;">庫存表（<code>inventory_tbl</code>）</h4>
  <ul>
    <li><strong>item_id</strong>（字串）：唯一產品標識（例如 SG001）。</li>
    <li><strong>name</strong>（字串）：太陽鏡款式（例如 Aviator、Round）。</li>
    <li><strong>description</strong>（字串）：產品的文字描述。</li>
    <li><strong>quantity_in_stock</strong>（整數）：當前可用庫存。</li>
    <li><strong>price</strong>（浮點）：以美元計價。</li>
  </ul>
  <h4 style="margin-top:1em; color:#1E40AF;">交易表（<code>transactions_tbl</code>）</h4>
  <ul>
    <li><strong>transaction_id</strong>（字串）：唯一交易標識（例如 TXN001）。</li>
    <li><strong>customer_name</strong>（字串）：顧客姓名，或期初條目的 <code>OPENING_BALANCE</code>。</li>
    <li><strong>transaction_summary</strong>（字串）：交易的簡短描述。</li>
    <li><strong>transaction_amount</strong>（浮點）：該筆交易的金額。</li>
    <li><strong>balance_after_transaction</strong>（浮點）：應用該交易後的餘額。</li>
    <li><strong>timestamp</strong>（字串）：ISO-8601 格式的交易日期/時間。</li>
  </ul>
</div>


## 結合程式碼執行的規劃

### 2.1. 計劃

在明確了模式後，你將構建一個**提示**，使模型透過*編寫程式碼來規劃*，並隨後執行該程式碼。核心理念是：程式碼即計劃。模型在註釋中解釋每一步，然後執行它。下面的提示還讓模型自行判斷請求是隻讀還是狀態變更，並強制安全執行（無 I/O、無網路、僅 TinyDB 查詢、變更一致）。


In [ ]:
PROMPT = """你是一名高階資料助手。透過編寫 TINDYDB PYTHON 程式碼來制定計劃。

資料庫模式和示例 (只讀):
{schema_block}

執行環境 (已匯入/已提供):
- 變數: db, inventory_tbl, transactions_tbl  # TinyDB 表物件
- 助手函式: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
- 自然語言: user_request: str  # 原始使用者訊息

規劃規則 (關鍵):
- 從 user_request 派生所有過濾器/引數 (形狀/關鍵詞, 價格範圍 "低於/高於/之間", 庫存提及, 數量, 購買/退貨意圖)。不要硬編碼值。
- 使用 Query() 動態構建 TinyDB 查詢。如果 user_request 中沒有某個約束，則不要應用它。
- 保持保守：如果意圖不明確，請執行只讀操作 (演習 - DRY RUN)。

交易政策 (硬性規定):
- 不要建立聚合的多專案交易。
- 如果請求包含多個專案，則為每個專案建立單獨的交易行。
- 對於每個專案：
  - 計算其自己的單行總計 (unit_price * qty),
  - 插入一筆該金額的交易,
  - 按順序更新餘額 (balance += line_total),
  - 更新該專案的庫存。
- 如果任何請求的專案庫存不足，不要更改任何資料；回覆 STATUS="insufficient_stock"。

人類可讀響應要求 (硬性規定):
- 你必須設定一個名為 `answer_text` (str 型別) 的變數，內容為簡短、客戶友好的句子 (1-2 行)。
- 這個句子是唯一面向使用者的訊息。沒有資料幀/JSON，沒有樣板式的免責宣告。
- 如果沒有匹配項，禮貌地說明情況，並提供一個相近的替代方案 (最接近的款式/價格) 或下一步建議。

行動政策:
- 如果請求明確要求改變狀態 (購買/採購/退貨/補貨/調整):
    ACTION="mutate"; SHOULD_MUTATE=True; 執行更改並寫入匹配的交易行。
  否則:
    ACTION="read"; SHOULD_MUTATE=False; 模擬並作為演習 (dry run) 簡要說明 (僅在日誌中)。

失敗與邊緣情況處理 (必須實現):
- 不要在 Query.test 中捕獲外部變數。將它們作為顯式引數傳遞。
- 始終設定一個簡短的 `answer_text`。同時設定一個字串 `STATUS`，其值為以下之一:
  "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent"。
- no_match: 沒有專案滿足過濾器 → 建議風格/價格最接近的，或邀請客戶提供不同範圍。
- insufficient_stock: 找到專案但庫存 < 請求數量 → 說明可用數量，並提供你能滿足的最大數量。
- invalid_request: 無法解析基本資訊 (例如 購買/退貨 的數量) → 簡潔地詢問缺失的部分。
- unsupported_intent: 該行動超出了商店的能力範圍 → 提供最接近的支援替代方案。
- 在所有情況下，保持樂於助人且簡潔的語氣 (1-2 句話)。僅在 stdout 日誌中放入技術細節 (例如 ACTION/DRY RUN)。

輸出契約:
- 僅在這些標籤之間返回可執行的 Python (沒有額外文字):
  <execute_python>
  # 你的 python
  </execute_python>

程式碼核對清單 (在程式碼中遵循):
1) 從 user_request 解析意圖和約束 (可用正規表示式)。
2) 漸進式構建 TinyDB 條件；查詢 inventory_tbl。
3) 如果是變更(mutate)操作: 驗證庫存, 更新庫存, 插入一筆交易 (新 id, 金額, 餘額, 時間戳)。
4) 始終設定:
   - `answer_text` (人類可讀的句子, 必需),
   - `STATUS` (參見上面的列表)。
   同時向 stdout 列印一個簡短的日誌, 例如 "LOG: ACTION=read DRY_RUN=True STATUS=no_match"。
5) 可選: 如果有用的話設定 `answer_rows` 或 `answer_json`，但 `answer_text` 是強制性的。

語氣示例 (用於 `answer_text`):
- 成功: "是的，我們有經典款太陽鏡，這是一款圓形鏡框，售價60美元。"
- 無匹配: "我們目前沒有100美元以下的圓形鏡框庫存，但我們的 Moon圓形鏡框有貨，售價120美元。"
- 庫存不足: "Classic款我們只剩下1副了；我可以為您預留。"
- 無效請求: "我可以處理這個——請問您想購買多少副？"
- 不支援的意圖: "我們不能翻新鏡框，但我可以推薦類似的新款式。"

約束條件:
- 使用 TinyDB Query 進行過濾。僅在需要時使用標準庫匯入。
- 保持程式碼清晰，並使用編號步驟進行註釋。

使用者請求:
{question}
"""

### 2.2 從提示到程式碼（在程式碼中規劃）

讓我們生成一段**即為計劃本身**的程式碼。

與其讓模型輸出 JSON 計劃並用許多小工具逐步執行，不如讓它**編寫一段涵蓋整個計劃的 Python 程式碼**（例如，“先過濾這個，再計算那個，然後更新該行”）。函式 `generate_llm_code`：

1. 從 `inventory_tbl` 與 `transactions_tbl` **構建實時模式**，讓模型看到真實欄位、型別與示例。
2. 使用該模式和使用者問題**格式化提示**。
3. **呼叫模型**生成**帶程式碼的計劃**響應——通常是一個 `<execute_python>...</execute_python>` 程式碼塊，內部包含逐步邏輯。
4. **返回完整響應**（包含計劃與程式碼）。  
   *此步不執行任何程式碼。*

為何採用此模式？充分利用模型已“熟悉”的 Python/TinyDB 作為豐富工具箱，使其能直接在程式碼中組合多步解法，而不是依賴不斷膨脹的定製工具集。我們將在後續步驟中提取並執行程式碼。

In [ ]:
# ---------- 1) 程式碼生成 ----------
def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.2,
) -> str:
    """
    請求 LLM 生成一個“帶程式碼的計劃”響應。
    返回完整的助手內容 (包括周圍的文字和標籤)。
    實際的程式碼提取稍後在 execute_generated_code 中進行。
    """
    schema_block = inv_utils.build_schema_block(inventory_tbl, transactions_tbl)
    prompt = PROMPT.format(schema_block=schema_block, question=prompt)

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "你編寫安全、註釋良好的 TinyDB 程式碼來處理資料問題和更新。"
            },
            {"role": "user", "content": prompt},
        ],
    )
    content = resp.choices[0].message.content or ""
    
    return content

### 2.3 嘗試一個示例提示（在程式碼中規劃）

我們使用以下提示作為示例：

> **提示：** “你們是否有庫存中的圓形太陽鏡，且價格低於 100 美元？”

在生成程式碼之前，先人工檢查 TinyDB 表，確認是否確有*圓形*框（僅按詞匹配）以及價格情況。執行下一個單元以預覽庫存，並突出顯示與“round”詞匹配的條目。

In [ ]:
Item = Query()                  # 建立一個 Query 物件來引用欄位 (例如 Item.name, Item.description)

# 搜尋庫存表，查詢描述(description)或名稱(name)
# 包含單詞 "round" (不區分大小寫) 的文件。檢查是內聯完成的：
# - (v or "") 確保我們透過將 None 轉換為空字串來處理它
# - .lower() 規範化大小寫
# - " round " 強制執行一個粗略的單詞邊界 (不會匹配 "wraparound")
round_sunglasses = inventory_tbl.search(
    (Item.description.test(lambda v: " round " in ((v or "").lower()))) |
    (Item.name.test(        lambda v: " round " in ((v or "").lower())))
)

# 在 notebook UI 中將結果渲染為格式化的 JSON
utils.print_html(json.dumps(round_sunglasses, indent=2), title="Inventory Status: Round Sunglasses")

好的——我們確實有圓形鏡框。根據我們的人工檢查，有兩款圓形鏡框有庫存，但只有**一款**價格**低於100美元**。因此，滿足該要求的商品是：

```python
{
  "item_id": "SG005",
  "name": "Classic",
  "description": "經典的圓形輪廓，搭配簡約的金屬鏡框，提供永恆而百搭的風格，適合休閒和正式場合穿搭。",
  "quantity_in_stock": 10,
  "price": 60
}
```

現在，讓我們要求模型**生成一個程式碼計劃**來回答這個提示（先不執行）。

In [ ]:
# 示例提示
prompt_round = "你們是否有庫存中的圓形太陽鏡，且價格低於 100 美元？"

# 生成以程式碼為計劃的響應（完整內容；可能包含 <execute_python> 標籤）
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)

# 檢視 LLM 的計劃與程式碼（此處不執行）
utils.print_html(full_content_round, title="以程式碼為計劃（完整響應）")

### 2.4. 定義執行器函式（執行給定計劃）

我們將定義一個函式，**安全地接收模型生成的計劃並執行**：

- 它**既可接收**完整的 LLM 響應（包含 `<execute_python>…</execute_python>`），**也可接收**原始 Python 程式碼。
- 需要時**提取**可執行程式碼塊。
- 在**受控名稱空間**中執行程式碼（僅 TinyDB 表與安全輔助函式）。
- 捕獲**標準輸出**、**錯誤**以及模型設定的答案變數（`answer_text`、`answer_rows`、`answer_json`）。
- 渲染**前/後**錶快照，明確展示副作用。

這是將**以程式碼為計劃**的結果轉化為行動與簡潔使用者答案的“執行器”。


In [ ]:
# --- 輔助函式：提取 <execute_python>...</execute_python> 之間的程式碼 ---
def _extract_execute_block(text: str) -> str:
    """
    返回 <execute_python>...</execute_python> 標籤內的 Python 程式碼。
    如果未找到標籤，則假定 'text' 已經是原始 Python 程式碼。
    """
    if not text:
        raise RuntimeError("空內容被傳遞給程式碼執行器。")
    m = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


# ---------- 2) 程式碼執行 ----------
def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    """
    在受控的名稱空間中執行程式碼。
    接受原始 Python 程式碼 或 包含 <execute_python> 標籤的完整內容。
    返回最小化的產物：stdout、error 和提取的答案。
    """
    # 在此處提取程式碼 (現在是集中處理)
    code = _extract_execute_block(code_or_content)

    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }

    # 捕獲從被執行程式碼產生的 stdout
    _stdout_buf, _old_stdout = io.StringIO(), sys.stdout
    sys.stdout = _stdout_buf
    err_text = None
    try:
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)
    except Exception:
        err_text = traceback.format_exc()
    finally:
        sys.stdout = _old_stdout
    printed = _stdout_buf.getvalue().strip()

    # 提取由生成程式碼設定的可能答案
    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )


    return {
        "code": code,           # <- 已經沒有標籤了
        "stdout": printed,
        "error": err_text,
        "answer": answer,
        "transactions_tbl": transactions_tbl.all(), # 供檢查
        "inventory_tbl": inventory_tbl.all(),     # 供檢查
    }

你已檢查庫存並確認：低於 100 美元的圓形款式恰有一款。現在到有趣的部分：將模型的以程式碼為計劃交給執行器，看看它如何完成工作。執行器會提取 <code><execute_python>...</execute_python></code> 程式碼塊，在受限沙箱中執行，並展示所有關鍵資訊——表發生了什麼變化（前/後）、計劃列印的日誌，以及最終的、面向使用者的簡潔 answer_text。

In [ ]:
# 針對圓形太陽鏡問題執行生成的計劃
result = execute_generated_code(
    full_content_round,          # 先前生成的完整 LLM 響應
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round,   # 例如："你們是否有庫存中的圓形太陽鏡，且價格低於 100 美元？"
)

# 檢視計劃實際執行的 Python 所提取的答案
utils.print_html(result["answer"], title="計劃執行 · 提取的答案")

可以看到，這與我們先前的人工分析所得結果一致。

## 2.4 退回兩副 Aviator 太陽鏡

在先前步驟中我們只是**查詢**資料，因此庫存與交易未發生變化。  
現在使用在程式碼中規劃的模式來處理一個**退貨**場景：
> **請求：** “退回我上週購買的 2 副 Aviator 太陽鏡。”

在生成計劃之前，先**檢查當前庫存**中的 *Aviator* 款。

In [ ]:
Item = Query()                    # 建立 Query 物件以引用欄位（例如 Item.name、Item.description）

# 查詢：提取所有 'name' 恰為 "Aviator" 的庫存行。
# 說明：
# - 這是區分大小寫的相等判斷。"aviator" 不會匹配。
# - 若需不區分大小寫匹配，可考慮 .test(...) 或 .matches(..., re.I)。
aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

# 以可讀的 JSON 面板展示匹配文件
utils.print_html(json.dumps(aviators, indent=2), title="庫存狀態：退貨前的 Aviator 太陽鏡")

庫存顯示一款 Aviator SKU 在庫——**SG001 (Aviator)**：**23** 件，每件 **$80**。現在生成一個計劃來回答該請求：

In [ ]:
prompt_aviator = "退回我上週購買的 2 副 Aviator 太陽鏡。"

# 生成以程式碼為計劃的響應（完整內容；可能包含 <execute_python> 標籤）
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1,
)

# 檢視 LLM 的計劃與程式碼（此處不執行）
utils.print_html(full_content_aviator, title="以程式碼為計劃（完整響應）")

在執行計劃之前，先檢查當前的交易狀態。

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="退貨前的交易表")

當前交易日誌僅顯示一條記錄——期初餘額（`TXN001`）為 `$500.00`，記錄時間 `2025-10-03T09:16:59.628898`。 

準備就緒——執行下方單元以執行計劃。

In [ ]:
# 執行針對 Aviator 太陽鏡退貨的生成計劃
result = execute_generated_code(
    full_content_aviator,          # 先前生成的完整 LLM 響應
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator,   # 例如："退回我上週購買的 2 副 Aviator 太陽鏡。"
)

# 檢視計劃實際執行的 Python 所提取的答案
utils.print_html(result["answer"], title="計劃執行 · 提取的答案")

如下所示，已為 Aviator 太陽鏡退貨插入一條新交易記錄。

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="退貨後的交易表")

執行下方單元后，你將看到 Aviator 的庫存增加到 25（`quantity_in_stock`）。

In [ ]:
Item = Query()                  

aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

utils.print_html(json.dumps(aviators, indent=2), title="庫存狀態：退貨後的 Aviator 太陽鏡")

## 3. 綜合起來：客服智慧體

你已構建各個元件——模式、提示、程式碼生成器與執行器。現在將它們串聯為一個助手：接收自然語言請求，生成以程式碼為計劃，安全執行，並展示結果（含前/後錶快照）。

**該智慧體的功能**
- 可選地重置演示資料以獲得乾淨執行。
- 生成計劃（Python 位於 `<execute_python>…</execute_python>` 中）。
- 在受控名稱空間中執行計劃（TinyDB + 輔助函式）。
- 輸出簡潔的 `answer_text` 並渲染前/後快照。

In [ ]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "o4-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    """
    端到端助手：
      1) (可選) 重新填充庫存和交易資料
      2) 根據 `question` 生成“程式碼即計劃”
      3) 在受控的名稱空間中執行
      4) 渲染執行前後的快照並返回產物

    Returns:
      {
        "full_content": <原始 LLM 響應 (可能包含 <execute_python> 標籤)>,
        "exec": {
            "code": <提取出的 python 程式碼>,
            "stdout": <計劃日誌>,
            "error": <回溯資訊或 None>,
            "answer": <答案文字/行/json>,
            "inventory_after": [...],
            "transactions_after": [...]
        }
      }
    """
    # 0) 可選的重新填充資料
    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    # 1) 顯示問題
    utils.print_html(question, title="使用者問題")

    # 2) 生成“程式碼即計劃”(完整內容)
    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )
    utils.print_html(full_content, title="以程式碼為計劃（完整響應）")

    # 3) 執行前的快照
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="庫存表 · 之前")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="交易表 · 之前")

    # 4) 執行
    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    # 5) 執行後的快照 + 最終答案
    utils.print_html(exec_res["answer"], title="計劃執行 · 提取的答案")
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="庫存表 · 之後")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="交易表 · 之後")

    # 6) 返回產物
    return {
        "full_content": full_content,
        "exec": {
            "code": exec_res["code"],
            "stdout": exec_res["stdout"],
            "error": exec_res["error"],
            "answer": exec_res["answer"],
            "inventory_after": inventory_tbl.all(),
            "transactions_after": transactions_tbl.all(),
        },
    }

## 4. 試用（藉助客服智慧體）

使用 `customer_service_agent(...)` 助手，從自然語言請求 → 以程式碼為計劃 → 安全執行 → 前/後快照。

**試試這些提示：**
1）**只讀（上面的示例）：**  
   “你們是否有庫存中的圓形太陽鏡，且價格低於 100 美元？”
2）**變更——退貨：**  
   “退回 2 副 Aviator 太陽鏡。”
3）**變更——購買：**  
   “為顧客 Alice 購買 3 副 Wayfarer 太陽鏡。”
4）**變更——購買多件：**
   “我想買 3 副 Classic 太陽鏡和 1 副 Aviator。”


<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:8px; padding:14px 16px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
  🔎 <strong><code>reseed=True</code> 有什麼作用？</strong><br><br>
  呼叫 <code>customer_service_agent(..., reseed=True)</code> 時，智慧體會在執行提示前<em>重新初始化</em>演示資料：
  <ul style="margin:8px 0 0 18px;">
    <li><strong>重置</strong> <code>inventory_tbl</code> 為預設產品集合。</li>
    <li><strong>重置</strong> <code>transactions_tbl</code> 為單一的期初餘額記錄。</li>
    <li>確保一次<strong>乾淨、可復現</strong>的執行，避免受先前測試影響。</li>
  </ul>
  若希望<strong>保留</strong>當前狀態並在既有操作上繼續，請設定 <code>reseed=False</code>。
</div>



In [ ]:
prompt = "我想購買 3 副 Classic 太陽鏡和 1 副 Aviator 太陽鏡。"

out = customer_service_agent(
    prompt,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
    reseed=True,   # 設為 False 以保留當前庫存與交易狀態
)

## 5. 要點總結

- **讓程式碼成為計劃。** 遵循“以程式碼為行動”的理念，讓模型編寫串聯步驟的 Python（過濾 → 計算 → 更新），然後直接執行。

- **避免脆弱的工具拼湊。** 不再堆砌小工具或 JSON 計劃，而是使用 Python/TinyDB——為模型提供一個熟悉且強大的工具箱，用同一個提示處理多種查詢形態。

- **保持執行安全且可見。** 在受控名稱空間中執行、捕獲日誌/錯誤，並檢視前/後表——因此你始終清楚變化是什麼以及原因。

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>

你已完成本範例並構建了一個<em>具備能動性</em>的客服工作流。你讓模型用程式碼作為計劃、安全執行，並用簡單的校驗保證更新可靠。發生失敗時，能給出清晰、可讀的原因；成功時，透過前/後快照看到具體變化。

藉助該模式——在程式碼中規劃並透明執行——你已準備好設計屬於自己的工作流：自動化、安全且易擴充套件。🚀

</div>
